In [3]:
# import statements

# import PyPhenom as ppi
import numpy as np
import matplotlib.pyplot as plt
import cv2

from math import factorial
from scipy.fft import fft2, fftshift
from scipy.signal import fftconvolve
from scipy.spatial.distance import pdist, squareform
from lmfit import Model, create_params
from skopt import gp_minimize
from skimage.draw import polygon

%matplotlib widget


c:\ProgramData\anaconda3\Lib\site-packages\paramiko\transport.py:219: CryptographyDeprecationWarning: Blowfish has been deprecated
  "class": algorithms.Blowfish,


In [ ]:
# connects to SEM 

PhenomID = 'MVE086101-20057-F'
username = 'MVE08610120057F'
password = 'KN68X6HJQMT9'

phenom = ppi.Phenom(PhenomID, username, password)

In [ ]:
# misc helper functions

def take_image():
    """
    Takes an image of the current location shown on the SEM and returns the image array
    and the pixel dimensions (width and height) in meters.
    """

    xdim = 256 # x and y dimensions of image taken
    ydim = 256
    stack = 1
    HDR = False
    scale = 1. #Set to a value < 1 to zoom in on a part of the image - centered

    phenom.SemAutoContrastBrightness()
    acq = phenom.SemAcquireImage(xdim, ydim, stack, ppi.DetectorMode.All, HDR, scale)

    img_array = np.array(acq.image) # converts image into an array
    width = acq.metadata.pixelSize.width # gets the pixel width
    height = acq.metadata.pixelSize.height # gets the pixel height

    return img_array, width, height


def calibrate_kernel_size(type, val_min):
    """
    Returns the kernel size used to apply a blur to the images taken while focusing the SEM.

    Parameter type: a string ("wd", "xst", "yst") representing the parameter being optimized
    Parameter val_min: the minimum working distance, x-stigmate, or y-stigmate (float) 
                       that the SEM can image at
    """

    sums = -1
    kernel_size = 5

    # sets magnification to 10000 for working distance calibration and the working distance to val_min
    if type == 'wd':
        phenom.SetHFW(ppi.MagnificationToFieldWidth(10000, 0.52))
        phenom.SetSemWD(val_min)

    # sets magnification to 40000 for stigmate calibration and the x-stigmate to val_min
    elif type == "xst":
        phenom.SetHFW(ppi.MagnificationToFieldWidth(40000, 0.52))
        phenom.SetSemStigmate(ppi.Position(val_min, 0))

    # sets magnification to 40000 for stigmate calibration and the y-stigmate to val_min
    elif type == "yst":
        phenom.SetHFW(ppi.MagnificationToFieldWidth(40000, 0.52))
        phenom.SetSemStigmate(ppi.Position(0, val_min))

    else:
        phenom.SetHFW(ppi.MagnificationToFieldWidth(40000, 0.52))
        phenom.SetSemStigmate(ppi.Position(val_min, val_min))

    img_array, width, height = take_image() # takes image of current location

    while np.round(sums, 4) != 0:
        blurred_img = cv2.GaussianBlur(img_array, (kernel_size, kernel_size), 0) # blurs the image
        filtered_data = cv2.Canny(blurred_img, 50, 150) # applies a canny filter to the blurred image

        sums = filtered_data.sum() # sums up the pixel values in the filtered image
        
        if np.round(sums, 4) != 0: # if the sum is not 0, add 2 to the kernel size and repeat
            kernel_size = kernel_size + 2

    phenom.SetSemStigmate(ppi.Position(0, 0)) # sets the stigmate back to (0, 0) for autofocusing

    return kernel_size


def reposition():
    """
    Repositions the SEM so that there is a feature in the center of the imaged location.
    """

    img_array, width, height = take_image() # takes picture of current location
    blurred_img = cv2.GaussianBlur(img_array, (size, size), 0) # blurs the image array
    filtered = cv2.Canny(blurred_img, 50, 150) # applies a canny filter

    # calculates the distance between each edge pixel in the filtered image and the other edge pixels
    points = [(i, j) for i, row in enumerate(filtered) for j, val in enumerate(row) if val == 255]
    distances = squareform(pdist(points, metric='euclidean'))
    total_distances = [sum(row) for row in distances]

    # selects the pixel with the smalleset distance between other pixels
    new_center = (points[np.argmin(total_distances)][1], points[np.argmin(total_distances)][0])

    # moves to the new center
    center = (img_array.shape[0] // 2, img_array.shape[1] // 2)
    px_distance = [center[0] - new_center[0], center[1] - new_center[1]]
    m_distance = [px_distance[0] * width, px_distance[1] * height]
    phenom.MoveBy(-1 * m_distance[0], m_distance[1])

In [ ]:
# global variables
wd_val_min = 0.002 #0.008 #0.002
wd_val_max = 0.012 #0.011 #0.025 #0.12198 is max value
xst_val_min = -0.2 #-1
xst_val_max = 0.2 #1
yst_val_min = -0.2 #-1
yst_val_max = 0.2 #1

size = calibrate_kernel_size('wd', wd_val_min)
x_size = calibrate_kernel_size('xst', xst_val_min)
y_size = calibrate_kernel_size('yst', yst_val_min)
stig_size = calibrate_kernel_size('stig', xst_val_min)

In [ ]:
# autocorrelation functions

def auto_correlation(frame):
    """
    Returns the computed auto_correlation for a given frame.

    Parameter frame: a Numpy array representing an SEM image
    """

    ft = np.fft.fft2(frame)
    autocorr = np.fft.ifft2(ft * np.conjugate(ft))
    return np.fft.fftshift(autocorr).real


def filtered_autocorr(img_array, kernel_size):
    """
    Returns the autocorrelation array for the given image array after applying a blur 
    (given the blurring kernel kernel_size) and a canny filter to img_array.

    Parameter img_array: a Numpy array representing an image
    Parameter kernel_size: an int specifying the size of the blurring kernel
    """
    
    blurred_img = cv2.GaussianBlur(img_array, (kernel_size, kernel_size), 0) # blurs the image
    filtered_data = cv2.Canny(blurred_img, 50, 150) # sets the values of edges = 255, everything else = 0
    acdata = auto_correlation(filtered_data) # calculates the auto_correlation data

    return acdata

In [ ]:
# fitting functions

# Define basic Zernike polynomials (on the unit disk)
def zernike_radial(n, m, rho):
    R = np.zeros_like(rho)
    for k in range((n - abs(m)) // 2 + 1):
        coeff = (-1)**k * factorial(n - k)
        coeff /= (factorial(k) *
                  factorial((n + abs(m)) // 2 - k) *
                  factorial((n - abs(m)) // 2 - k))
        R += coeff * rho**(n - 2 * k)
    return R


def zernike(n, m, rho, theta):
    if m > 0:
        return zernike_radial(n, m, rho) * np.cos(m * theta)
    elif m < 0:
        return zernike_radial(n, -m, rho) * np.sin(-m * theta)
    else:
        return zernike_radial(n, 0, rho)


# Mapping Noll index to (n, m)
NOLL_MAP = {
    1:  (0,  0),   # Piston (constant phase offset)
    2:  (1, -1),   # Tilt X (Tip)
    3:  (1,  1),   # Tilt Y (Tilt)
    4:  (2, -2),   # Oblique Astigmatism
    5:  (2,  0),   # Defocus
    6:  (2,  2),   # Vertical Astigmatism
    7:  (3, -3),   # Vertical Trefoil
    8:  (3, -1),   # Vertical Coma
    9:  (3,  1),   # Horizontal Coma
    10: (3,  3),   # Horizontal Trefoil
    11: (4, -4),   # Oblique Tetrafoil
    12: (4, -2),   # Oblique Secondary Astigmatism
    13: (4,  0),   # Primary Spherical Aberration
    14: (4,  2),   # Vertical Secondary Astigmatism
    15: (4,  4),   # Vertical Tetrafoil
}


def incoherent_zernike_psf(N=256, defocus = None, stig = None): # zernike_coeffs=None):
    if defocus is None: # if zernike_coeffs is None:
        zernike_coeffs = {
            5:2,  # defocus
        }

    else:
        zernike_coeffs = {
            5:defocus
        }

    if stig is not None:
        zernike_coeffs[6] = stig

    x = np.linspace(- 1, 1, N)
    X, Y = np.meshgrid(x, x)
    rho = np.sqrt(X**2 + Y**2)
    theta = np.arctan2(Y, X)

    aperture = rho <= 1 #boolean array

    wavefront = np.zeros_like(rho)

    for j, coeff in zernike_coeffs.items():
        n, m = NOLL_MAP[j]
        Z = zernike(n, m, rho, theta)
        Z[~aperture] = 0
        wavefront += coeff * Z

    # Pupil function with aberration
    phase = 2 * np.pi * wavefront # this could be scaled by the wavelength.
    # suggest to not to do this in this case that adds to the confusion ==> this is not the same as a coherent point source!

    pupil = np.exp(1j * phase) 
    pupil[~aperture] = 0

    # Coherent PSF
    h = fftshift(fft2(pupil)) #The point spread function (PSF), which is given by the square of the absolute value of the 
    psf_coh = np.abs(h)**2    #Fourier transform of the wavefront, shows what the intensity image of a point source would look like.

    # Incoherent PSF: autocorrelation of coherent intensity
    psf_incoh = fftconvolve(psf_coh, psf_coh, mode='same')
    psf_incoh /= psf_incoh.sum()

    return psf_incoh


def incoh_psf_fit(ac, df, st):
    model = Model(incoherent_zernike_psf)

    nrows = ac.shape[0]
    ncols = ac.shape[1]

    pars = create_params(N = dict(value = 256, vary = False),
                         defocus = dict(value = df, vary = True, min = 0),
                         stig = dict(value = st, vary = True))

    result = model.fit(ac, params = pars) # uses the above parameters and  model to fit the data

    # plt.figure()
    # plt.imshow(ac)
    # plt.contour(result.best_fit)

    return result

In [ ]:
def remove_background(acdata):
    """
    Returns the normalized autocorrelation with the background removed.

    Parameter acdata: a 2D Numpy array of the autocorrelation of an image
    """

    contourplot = plt.contour(acdata) # creates the contour plot

    contour_lines = []
    for collection in contourplot.collections:
        for path in collection.get_paths():
            contour_lines.append(path.vertices) # gets the coords of the contour lines

    filtered = np.zeros((acdata.shape[0], acdata.shape[1], 3), dtype=np.uint8) # generates a black image

    all_ellipses = []
    for level in contour_lines:
        ellipse_pts = []

        x_sum = 0
        y_sum = 0
        for px in level:
            x_sum += px[0] 
            y_sum += px[1]
        x_avg = x_sum / len(level) # calculates the midpt of the contour level
        y_avg = y_sum / len(level)

        if (round(x_avg, -1) == round((acdata.shape[1] / 2), -1) and 
            round(y_avg, -1) == round((acdata.shape[0] / 2), -1)):
            for px in level:
                ellipse_pts.append(px) # selects the contour levels that are centered around the middle of the ac
        all_ellipses.append(ellipse_pts)

    largest_ellipse = max(all_ellipses, key=len) # finds the largest ellipse

    x = []
    y = []
    for pt in largest_ellipse:
        x.append(pt[1])
        y.append(pt[0])
    row, col = polygon(x, y) # all the coords inside of the ellipse
    filtered[row, col] = 255 # fill the ellipse
    grayscale_img = cv2.cvtColor(filtered, cv2.COLOR_BGR2GRAY) # converts the filtered image to grayscale

    threshold = acdata[int(largest_ellipse[0][1]), int(largest_ellipse[0][0])] # sets a threshold equal to
    mask = grayscale_img == 0                                                 # a pt on the edge of the ellipse
    acdata[mask] = threshold # img_array has a val equal to threshold where grayscale_img has a val of 0

    for r in range(acdata.shape[0]):
        for c in range(acdata.shape[1]): # substracts either the threshold or img_array value, whichever is smaller
            acdata[r][c] = acdata[r][c] - min(threshold, acdata[r][c]) # from every point in img_array

    acdata = acdata / acdata.sum() # normalizes the data

    # plt.figure()
    # plt.imshow(acdata)
    # plt.colorbar()
    
    return acdata

In [ ]:
# optimization functions

def optimum_wd(wd, kernel_size = size):
    """
    Returns the negative sum of the autocorrelation values for the four pixels surrounding the 
    center pixel given wd, a list containing a working distance.
    This is the objective function to minimize the working distance.

    Parameter wd: a list containing an int or float specifying the working distance 
    Parameter kernel_size: an int specifying the size of the blurring kernel
    """

    phenom.SetSemWD(wd[0]) # sets the working distance to wd[0]

    img_array, width, height = take_image() # takes an image of the current location
    acdata = filtered_autocorr(img_array, kernel_size) # calculates the autocorrelation for the filtered image

    sums = (acdata[acdata.shape[0] // 2 - 1, acdata.shape[1] // 2] + 
            acdata[acdata.shape[0] // 2 + 1, acdata.shape[1] // 2] + 
            acdata[acdata.shape[0] // 2, acdata.shape[1] // 2 - 1] + 
            acdata[acdata.shape[0] // 2, acdata.shape[1] // 2 + 1]) # sums together the four center pixels
    
    return sums * (-1) # returns the negative of the sum


def auto_focus(wd_val_min, wd_val_max):
    """
    Determines the working distance corresponding to a focused image using bayesian optimization. 
    Sets the working distance to the value corresponding to a focused image after minimization.
    Returns the optimal working distance.

    Parameter wd_val_min: the minimum working distance (float) that is considered when minimizing
    Parameter wd_val_max: the maximum working distance (float) that is considered when minimizing
    """

    res = gp_minimize(optimum_wd, # calls scikit's bayesian optimization 
                      [(wd_val_min, wd_val_max)], 
                      acq_func = "EI", 
                      n_calls = 10, 
                      n_initial_points = 5, 
                      verbose = False, 
                      random_state = 1234) 

    phenom.SetSemWD(res.x[0]) # sets the working distance to the optimum
    return res.x[0]


def optimum_stig(st, kernel_size = stig_size):
    xst, yst = st
    phenom.SetSemStigmate(ppi.Position(xst, yst))

    img_array, width, height = take_image() # takes an image for the current location

    # acdata = filtered_autocorr(img_array, kernel_size) # calculates the autocorrelation for the filtered image

    # plt.subplot(1, 4, 4)
    # plt.imshow(acdata)
    # plt.colorbar()

    # sums = (acdata[acdata.shape[0] // 2 - 1, acdata.shape[1] // 2] + 
    #         acdata[acdata.shape[0] // 2 + 1, acdata.shape[1] // 2] + 
    #         acdata[acdata.shape[0] // 2, acdata.shape[1] // 2 - 1] + 
    #         acdata[acdata.shape[0] // 2, acdata.shape[1] // 2 + 1]) # sums together the four center pixels

    
    # return sums * (-1) # returns the negative of the sum

    acdata = auto_correlation(img_array)

    if acdata.sum() != 0:
        threshold = ((acdata.max() - acdata.min()) * 0.25) + acdata.min()
        acdata[acdata < threshold] = threshold
        acdata = acdata - threshold
        if acdata.sum() != 0:
            acdata /= acdata.sum()

    result = incoh_psf_fit(acdata, 1, 2)

    return result.best_values['stig']


def auto_stig(xst_val_min, xst_val_max, yst_val_min, yst_val_max):
    """
    Determines the stigmate values corresponding to a focused image using bayesian optimization. 
    Sets the working distance to the value corresponding to a focused image after minimization.

    Parameter xst_val_min: the minimum x-stigmate (float) that is considered when minimizing
    Parameter xst_val_max: the maximum x-stigmate distance (float) that is considered when minimizing
    Parameter yst_val_min: the minimum y-stigmate distance (float) that is considered when minimizing
    Parameter yst_val_max: the maximum y-stigmate distance (float) that is considered when minimizing
    """

    res = gp_minimize(optimum_stig, 
                      [(xst_val_min, xst_val_max), (yst_val_min, yst_val_max)], 
                      acq_func = "EI", 
                      n_calls = 30, 
                      n_initial_points = 5, 
                      verbose = False, 
                      random_state = 1234) 

    print(res.x[0], res.x[1])
    phenom.SetSemStigmate(ppi.Position(res.x[0], res.x[1]))

    return res.x[0], res.x[1]


def optimum_xstig(xst, kernel_size = x_size):
    """
    Returns the negative sum of the autocorrelation values for the four pixels surrounding the 
    center pixel given xst, a list containing a x-stigmate.
    This is the objective function to minimize the x-stigmate.

    Parameter xst: a list containing an int or float specifying the x-stigmate
    Parameter kernel_size: an int specifying the size of the blurring kernel
    """

    phenom.SetSemStigmate(ppi.Position(xst[0], 0)) # sets the x-stigmate to xst[0]

    img_array, width, height = take_image() # takes an image for the current location
    acdata = filtered_autocorr(img_array, kernel_size) # calculates the autocorrelation for the filtered image

    sums = (acdata[acdata.shape[0] // 2 - 1, acdata.shape[1] // 2] + 
            acdata[acdata.shape[0] // 2 + 1, acdata.shape[1] // 2] + 
            acdata[acdata.shape[0] // 2, acdata.shape[1] // 2 - 1] + 
            acdata[acdata.shape[0] // 2, acdata.shape[1] // 2 + 1]) # sums together the four center pixels
    
    return sums * (-1) # returns the negative of the sum

    acdata = auto_correlation(img_array)

    plt.figure()
    plt.subplot(1, 3, 1)
    plt.imshow(acdata)

    if acdata.sum() != 0:
        threshold = ((acdata.max() - acdata.min()) * 0.25) + acdata.min()
        acdata[acdata < threshold] = threshold
        acdata = acdata - threshold
        if acdata.sum() != 0:
            acdata /= acdata.sum()

    plt.subplot(1, 3, 2)
    plt.imshow(acdata)

    result = incoh_psf_fit(acdata, 1, 2)

    plt.subplot(1, 3, 3)
    plt.imshow(result.best_fit)

    return result.best_values['stig']


def optimum_ystig(yst, kernel_size = y_size):
    """
    Returns the negative sum of the autocorrelation values for the four pixels surrounding the 
    center pixel given yst, a list containing a y-stigmate.
    This is the objective function to minimize the y-stigmate.

    Parameter yst: a list containing an int or float specifying the y-stigmate
    Parameter kernel_size: an int specifying the size of the blurring kernel
    """

    phenom.SetSemStigmate(ppi.Position(0, yst[0])) # sets the x-stigmate to xst[0]

    img_array, width, height = take_image() # takes an image for the current location
    acdata = filtered_autocorr(img_array, kernel_size) # calculates the autocorrelation for the filtered image

    sums = (acdata[acdata.shape[0] // 2 - 1, acdata.shape[1] // 2] + 
            acdata[acdata.shape[0] // 2 + 1, acdata.shape[1] // 2] + 
            acdata[acdata.shape[0] // 2, acdata.shape[1] // 2 - 1] + 
            acdata[acdata.shape[0] // 2, acdata.shape[1] // 2 + 1]) # sums together the four center pixels
    
    return sums * (-1) # returns the negative of the sum

    acdata = auto_correlation(img_array)

    plt.figure()
    plt.subplot(1, 3, 1)
    plt.imshow(acdata)

    if acdata.sum() != 0:
        threshold = ((acdata.max() - acdata.min()) * 0.25) + acdata.min()
        acdata[acdata < threshold] = threshold
        acdata = acdata - threshold
        if acdata.sum() != 0:
            acdata /= acdata.sum()

    plt.subplot(1, 3, 2)
    plt.imshow(acdata)

    result = incoh_psf_fit(acdata, 1, 2)

    plt.subplot(1, 3, 3)
    plt.imshow(result.best_fit)

    return result.best_values['stig']


def auto_stig_sep(xst_val_min, xst_val_max, yst_val_min, yst_val_max):
    """
    Determines the stigmate values corresponding to a focused image using bayesian optimization. 
    Sets the working distance to the value corresponding to a focused image after minimization.

    Parameter xst_val_min: the minimum x-stigmate (float) that is considered when minimizing
    Parameter xst_val_max: the maximum x-stigmate distance (float) that is considered when minimizing
    Parameter yst_val_min: the minimum y-stigmate distance (float) that is considered when minimizing
    Parameter yst_val_max: the maximum y-stigmate distance (float) that is considered when minimizing
    """

    x_res = gp_minimize(optimum_xstig, 
                        [(xst_val_min, xst_val_max)], 
                        acq_func = "EI", 
                        n_calls = 15, 
                        n_initial_points = 5, 
                        verbose = False, 
                        random_state = 1234) 

    y_res = gp_minimize(optimum_ystig, 
                        [(yst_val_min, yst_val_max)], 
                        acq_func = "EI", 
                        n_calls = 15, 
                        n_initial_points = 5, 
                        verbose = False, 
                        random_state = 1234) 

    phenom.SetSemStigmate(ppi.Position(x_res.x[0], y_res.x[0]))

    return x_res.x[0], y_res.x[0]

In [ ]:
def thingy(wd_val_min, wd_val_max, xst_val_min, xst_val_max, yst_val_min, yst_val_max):
    phenom.SetHFW(ppi.MagnificationToFieldWidth(10000, 0.52))
    curr = auto_focus(wd_val_min, wd_val_max)
    reposition()

    for magnification in [20000, 40000]:
        phenom.SetHFW(ppi.MagnificationToFieldWidth(magnification, 0.52))
        auto_focus(curr - (1 / 1000), curr + (1 / 1000))
        reposition()

    phenom.SetHFW(ppi.MagnificationToFieldWidth(100000, 0.52))
    x, y = auto_stig(xst_val_min, xst_val_max, yst_val_min, yst_val_max)

    print('stig', x, y)

    for r in range(2):
        phenom.SetHFW(ppi.MagnificationToFieldWidth(80000, 0.52))
        auto_focus(curr - (1 / 1000), curr + (1 / 1000))

        phenom.SetHFW(ppi.MagnificationToFieldWidth(100000, 0.52))
        auto_stig(x - (0.1 / 20), x + (0.1 / 20), y - (0.1 / 20), y + (0.1 / 20))

thingy(wd_val_min, wd_val_max, xst_val_min, xst_val_max, yst_val_min, yst_val_max)